In [26]:
import json
from PIL import Image, ImageDraw, ImageFont
import subprocess

In [27]:
BASE_TRAILER_PATH = "/Users/sangdo/Downloads/movie-top-list/movie_trailer/"
img_bg_path = "/Users/sangdo/Downloads/movie-top-list/timeline_template/timeline_8000x1080.png"

In [28]:
#testing only
def generate_sample_timeline():
    WIDTH = 20000
    HEIGHT = 1080
    BLOCK_WIDTH = 600
    START_X = 400

    with open("data.json") as f:
        movies = json.load(f)

    img = Image.new("RGB",(WIDTH,HEIGHT),(15,15,15))
    draw = ImageDraw.Draw(img)

    title_font = ImageFont.truetype("Arial.ttf",70)
    text_font = ImageFont.truetype("Arial.ttf",40)

    draw.text((WIDTH/2-400,50),
            "Tom Cruise Movie Salaries Timeline",
            fill="white",
            font=title_font)

    timeline_y = HEIGHT//2

    draw.line((0,timeline_y,WIDTH,timeline_y),fill="white",width=6)

    x = START_X

    for m in movies:

        center = x + BLOCK_WIDTH//2

        draw.line((center,timeline_y-120,center,timeline_y+120),
                fill="white",width=5)

        draw.text((x,timeline_y-200),
                m["movie"],
                fill="white",
                font=text_font)

        draw.text((x,timeline_y-140),
                str(m["year"]),
                fill="gray",
                font=text_font)

        draw.text((x,timeline_y+150),
                m["salary"],
                fill="yellow",
                font=text_font)

        x += BLOCK_WIDTH + 120

    img.save("timeline.png")

    print("timeline.png created")

In [29]:
def generate_sample_video():
    duration = 480
    video_w = 1080

    positions = [0,800,1600,2400,3200,4000]

    pause = 3
    scroll_time = 8

    segments = []
    time = 0

    for i,p in enumerate(positions):

        segments.append(f"if(lt(t,{time+scroll_time}),{p},")
        time += scroll_time

        segments.append(f"if(lt(t,{time+pause}),{p},")
        time += pause

    segments.append("0" + ")"*len(segments))

    expr = "".join(segments)

    cmd = [ #
    "ffmpeg",
    "-loop","1",
    "-t",str(duration),
    "-i","timeline.png",
    "-vf",f"crop=1080:1920:x='{expr}':y=0",
    "-r","30",
    "-pix_fmt","yuv420p",
    "-c:v","libx264",
    "timeline_video.mp4"
    ]

    subprocess.run(cmd)

ffmpeg -loop 1 -t 480 -i timeline.png \
-vf "crop=1920:1080:x='(iw-1920)*t/480':y=0" \
-r 90 -pix_fmt yuv420p -c:v libx264 timeline_video.mp4

<h2>Download a part of video from youtube</h2>
#python3.10 -m pip install yt-dlp

yt-dlp \
--download-sections "*00:00:40-00:01:00" \
--force-keyframes-at-cuts \
-f "bv*+ba/b" \
--merge-output-format mp4 \
-o "maverick.mp4" \
"https://www.youtube.com/watch?v=qSqVVswa420"

<h2>Put a video at position X, Y inside a video</h2>

In [ ]:
def put_video_inside_video():
    # timeline canvas size (video size)
    canvas_width = 8000
    canvas_height = 1080

    # clip size (small clip same scale as 1920 x 1080)
    clip_width = 320
    clip_height = 180

    # scrolling speed
    scroll_speed = 120

    # movie nodes (file, x, y)
    movies = [
        (BASE_TRAILER_PATH + "tomcruise/maverick_small.mp4", 165, 150),   #position of the clip
        # ("clips/movie2.mp4", 800, 260),
        # ("clips/movie3.mp4", 1600, 260),
    ]

    inputs = ""
    filters = ""

    # build ffmpeg input list
    for i, (file, x, y) in enumerate(movies):
        inputs += f"-i {file} "
        filters += f"[{i+1}:v]scale={clip_width}:{clip_height},setpts=PTS-STARTPTS[v{i}];"

    # create base canvas
    filters += f"[0:v]scale={canvas_width}:{canvas_height}[base];"

    last = "base"

    # overlay clips
    for i, (_, x, y) in enumerate(movies):
        filters += f"[{last}][v{i}]overlay={x}:{y}:shortest=1[tmp{i}];"
        last = f"tmp{i}"

    # scrolling camera
    filters += f"[{last}]crop=1920:1080:x='t*{scroll_speed}':y=0[out]"
    
    # cmd = f'ffmpeg -loop 1 -i {img_bg_path} {inputs} -filter_complex "{filters}" -map "[out]" -t 90 -r 30 -c:v libx264 -preset veryfast -crf 22 -threads 0 -pix_fmt yuv420p tomcruise.mp4'
    cmd = f'ffmpeg -loop 1 -i {img_bg_path} {inputs} -filter_complex "{filters}" -map "[out]" -t 60 -r 30 -c:v libx264 -preset ultrafast -crf 23 -movflags +faststart -pix_fmt yuv420p tomcruise.mp4'
    print("Running FFmpeg...")
    subprocess.run(cmd, shell=True)

#test
put_video_inside_video()    #size w 8000: ~ 
#ffmpeg -i maverick.mp4 -vf scale=320:-1 maverick_small.mp4

Running FFmpeg...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, png_pipe, from '/Users/sangdo/Downloads/movie-top-list/timeline_template/timeline_8000x1080.png':
  Duration: N/A, bitrate: N/A
  Stream #0:0: Video: png, rgb24(pc, gbr/unknown/unknown), 8000x1